In [11]:

import json, re, math, textwrap
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# --- Helper: varimax rotation (orthogonal) ---
def varimax(Phi, gamma=1.0, q=20, tol=1e-6):
    """Perform varimax (orthogonal) rotation on loadings matrix.
    Returns rotated loadings and the rotation matrix.
    Phi: (n_features x n_factors)
    """
    p, k = Phi.shape
    R = np.eye(k)
    d = 0
    for i in range(q):
        d_old = d
        Lambda = np.dot(Phi, R)
        u, s, vh = np.linalg.svd(
            np.dot(
                Phi.T,
                (Lambda**3 - (gamma/p) * np.dot(Lambda, np.diag(np.sum(Lambda**2, axis=0))))
            )
        )
        R = np.dot(u, vh)
        d = np.sum(s)
        if d_old != 0 and d/d_old < 1 + tol:
            break
    return np.dot(Phi, R), R

# --- Load data ---
map_path = 'C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/big_five/map.json'
csv_path = 'C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/big_five/data-sample.csv'
with open(map_path, 'r') as f:
    var_map = json.load(f)

# Keep only item columns present in CSV and in map
item_cols = [col for col in var_map.keys()]

raw_df = pd.read_csv(csv_path)
# Some CSVs may include extra spaces in headers; strip
raw_df.columns = [c.strip() for c in raw_df.columns]

available_item_cols = [c for c in item_cols if c in raw_df.columns]
X = raw_df[available_item_cols].copy()

# Convert to numeric and coerce errors
for c in available_item_cols:
    X[c] = pd.to_numeric(X[c], errors='coerce')

# Replace obvious out-of-scale zeros with NaN if scale appears 1..5
# Heuristic: if a column's 0s exist and values 1..5 exist, treat 0 as missing.
for c in available_item_cols:
    series = X[c]
    if (series == 0).any() and series.dropna().between(1,5).any():
        X.loc[series == 0, c] = np.nan

# Impute missing by item median
X_imputed = X.fillna(X.median())

# Z-score standardization
scaler = StandardScaler()
Z = scaler.fit_transform(X_imputed.values)

# --- Parallel Analysis to pick number of factors ---
# Correlation matrix eigenvalues
corr = np.corrcoef(Z, rowvar=False)
vals, vecs = np.linalg.eig(corr)
emp_eigs = np.sort(np.real(vals))[::-1]

n_obs, n_vars = Z.shape
n_iter = 200
rand_eigs_mat = np.zeros((n_iter, n_vars))
for i in range(n_iter):
    # Generate random normal data with same shape
    rand = np.random.normal(size=(n_obs, n_vars))
    rand = (rand - rand.mean(axis=0)) / rand.std(axis=0, ddof=1)
    reig, _ = np.linalg.eig(np.corrcoef(rand, rowvar=False))
    rand_eigs_mat[i, :] = np.sort(np.real(reig))[::-1]

mean_rand_eigs = rand_eigs_mat.mean(axis=0)
# Best number of factors where empirical eigenvalues exceed mean random eigenvalues
best_factors = int(np.sum(emp_eigs > mean_rand_eigs))
if best_factors < 1:
    best_factors = 1
if best_factors > min(n_vars, 12):
    best_factors = min(n_vars, 12)

# --- Factor Analysis ---
fa = FactorAnalysis(n_components=best_factors, rotation=None, random_state=42)
fa_scores_unrot = fa.fit_transform(Z)  # (n_samples, n_factors)
loadings_unrot = fa.components_.T     # (n_vars, n_factors)

# Varimax rotation
loadings_rot, R = varimax(loadings_unrot)
# Rotate scores accordingly (orthogonal rotation)
fa_scores = fa_scores_unrot @ np.linalg.inv(R)

# Create loadings DataFrame
loadings_df = pd.DataFrame(loadings_rot, index=available_item_cols,
                           columns=[f'F{i+1}' for i in range(best_factors)])

# --- Name factors based on highest contributing Big Five prefixes ---
prefix_name_map = {
    'EXT': 'Extraversion',
    'EST': 'Neuroticism',  # IPIP label EST (Emotional Stability) ~ reverse of Neuroticism
    'AGR': 'Agreeableness',
    'CSN': 'Conscientiousness',
    'OPN': 'Openness'
}

def infer_factor_name(loadings_col, top_k=12):
    # Consider absolute top loadings
    s = loadings_col.abs().sort_values(ascending=False).head(top_k)
    # Tally by prefix
    tally = {}
    signed = {}
    for item, val in s.items():
        m = re.match(r'([A-Z]{3})', item)
        if not m:
            continue
        pref = m.group(1)
        tally[pref] = tally.get(pref, 0) + val
        signed[pref] = signed.get(pref, 0.0) + loadings_col[item]
    if not tally:
        return "General Factor"
    top_pref = max(tally.items(), key=lambda kv: kv[1])[0]
    base = prefix_name_map.get(top_pref, top_pref)
    # Determine orientation (positive vs negative) from signed contribution
    orientation = ' (high)' if signed.get(top_pref, 0) >= 0 else ' (low)'
    return base + orientation

factor_names = [infer_factor_name(loadings_df[c]) for c in loadings_df.columns]

# --- Cluster Analysis on Factor Scores ---
# Standardize factor scores for clustering
fs_scaler = StandardScaler()
FS = fs_scaler.fit_transform(fa_scores)

k_range = list(range(2, min(12, max(3, best_factors+5)) + 1))
ks, sils = [], []
models = {}
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=25)
    labels = km.fit_predict(FS)
    sil = silhouette_score(FS, labels)
    ks.append(k)
    sils.append(sil)
    models[k] = (km, labels)

best_k = ks[int(np.argmax(sils))]
best_model, best_labels = models[best_k]

# Cluster centroids in factor score space (unstandardized, for interpretability)
centroids_fs = fs_scaler.inverse_transform(best_model.cluster_centers_)
centroids_df = pd.DataFrame(centroids_fs, columns=[f'F{i+1}' for i in range(best_factors)])
centroids_df.index = [f'Cluster {i}' for i in range(best_k)]

# Map factor names to columns
name_map = {f'F{i+1}': factor_names[i] for i in range(best_factors)}

# Assign cluster to each person
assignments = pd.DataFrame({'Cluster': [f'Cluster {x}' for x in best_labels]})

# --- Cluster naming and descriptions ---

def label_level(x):
    if x >= 0.45:
        return 'high'
    elif x <= -0.45:
        return 'low'
    else:
        return 'moderate'

cluster_summaries = {}
for idx, row in centroids_df.iterrows():
    levels = {name_map[col]: label_level(val) for col, val in row.items()}
    # Construct a human-friendly name using pronounced highs and lows
    strong = [f"{trait.replace(' (high)','').replace(' (low)','')} {lvl}" 
              for trait, lvl in levels.items() if lvl != 'moderate']
    if strong:
        cl_name = ', '.join(strong)
    else:
        cl_name = 'Balanced profile'

    # Build three-sentence description
    overview_parts = [f"{t.replace(' (high)','').replace(' (low)','')} is {lvl}" 
                      for t, lvl in levels.items()]
    overview = 'Overall: ' + '; '.join(overview_parts) + '.'

    highs = [t.replace(' (high)','').replace(' (low)','') 
             for t, lvl in levels.items() if lvl == 'high']
    if highs:
        strengths = 'Strengths: typically strong on ' + ', '.join(highs) + '. '
    else:
        strengths = 'Strengths: none stand out markedly given the available factors. '

    lows = [t.replace(' (high)','').replace(' (low)','') 
            for t, lvl in levels.items() if lvl == 'low']
    if lows:
        weaknesses = 'Weaknesses: tends to be low on ' + ', '.join(lows) + '. '
    else:
        weaknesses = 'Weaknesses: no pronounced low areas given the available factors. '

    cluster_summaries[idx] = {
        'name': cl_name,
        'description': overview + ' ' + strengths + weaknesses
    }

# --- Person 18 summary ---
person_idx = 17  # zero-based index for Person 18
p18_scores = pd.Series(fa_scores[person_idx], index=[f'F{i+1}' for i in range(best_factors)])
p18_scores_z = pd.Series(FS[person_idx], index=[f'F{i+1}' for i in range(best_factors)])
p18_cluster = assignments.iloc[person_idx, 0]

# Determine strengths (scores >= +0.45), average (|z| < 0.45), weaknesses (<= -0.45)
strength_traits = [name_map[c].replace(' (high)','').replace(' (low)','') 
                   for c, v in p18_scores_z.items() if v >= 0.45]
weak_traits = [name_map[c].replace(' (high)','').replace(' (low)','') 
               for c, v in p18_scores_z.items() if v <= -0.45]
avg_traits = [name_map[c].replace(' (high)','').replace(' (low)','') 
              for c, v in p18_scores_z.items() if -0.45 < v < 0.45]

p18_summary = {
    'strengths': strength_traits,
    'average_or_weak': {
        'average': avg_traits,
        'weaknesses': weak_traits
    },
    'cluster': p18_cluster
}

# --------- PRINT KEY RESULTS ---------
print('Best number of factors (parallel analysis):', best_factors)

print('\nFactor names (after varimax):')
for i, n in enumerate(factor_names, 1):
    print(f'  F{i}: {n}')

print('\nTop 10 absolute loadings per factor:')
for c in loadings_df.columns:
    top = loadings_df[c].sort_values(key=lambda s: s.abs(), ascending=False).head(10)
    print(f'\n{c}:')
    for idx, val in top.items():
        print(f'  {idx}: {val:.3f}')

print('\nBest number of clusters (silhouette):', best_k)
print('\nCluster centroids in factor-score space:')
print(centroids_df.round(3))

print('\nCluster names and descriptions:')
for cl, info in cluster_summaries.items():
    print(f"\n{cl} — {info['name']}")
    wrapped = textwrap.fill(info['description'], width=100)
    print(wrapped)

print('\nPerson 18 factor z-scores:')
for c, v in p18_scores_z.items():
    print(f'  {c}: {v:.3f} ({name_map[c]})')
print('Assigned cluster:', p18_cluster)

print('\nPerson 18 quick summary object:')
print(p18_summary)


C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.p

Best number of factors (parallel analysis): 5

Factor names (after varimax):
  F1: Extraversion (high)
  F2: Agreeableness (high)
  F3: Neuroticism (low)
  F4: Openness (low)
  F5: Conscientiousness (high)

Top 10 absolute loadings per factor:

F1:
  EXT5: 0.827
  EXT7: 0.787
  EXT1: 0.744
  EXT3: 0.738
  EXT10: -0.728
  EXT2: -0.724
  EXT4: -0.659
  EXT9: 0.634
  EXT6: -0.620
  EXT8: -0.585

F2:
  AGR5: -0.757
  AGR4: 0.753
  AGR9: 0.743
  AGR7: -0.738
  AGR8: 0.642
  AGR2: 0.624
  AGR6: 0.552
  AGR1: -0.483
  AGR10: 0.438
  AGR3: -0.365

F3:
  EST6: -0.792
  EST9: -0.772
  EST8: -0.729
  EST7: -0.710
  EST1: -0.622
  EST3: -0.593
  EST2: 0.554
  EST5: -0.510
  EST10: -0.502
  AGR3: -0.399

F4:
  OPN10: -0.727
  OPN3: -0.675
  OPN4: 0.632
  OPN5: -0.563
  OPN1: -0.539
  OPN6: 0.539
  OPN8: -0.529
  OPN2: 0.522
  OPN7: -0.506
  CSN3: -0.429

F5:
  CSN6: -0.694
  CSN4: -0.663
  CSN5: 0.644
  CSN2: -0.595
  CSN8: -0.571
  CSN7: 0.530
  CSN1: 0.526
  CSN10: 0.472
  CSN9: 0.470
  CSN3: 0.4

C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


Strengths: Person 18 is distinctly Extraverted (~ +1.0 σ) with a secondary tilt toward Agreeableness (~ +0.46 σ), pointing to an energetic, sociable style that cooperates readily.
Average/weak: They are very low on Conscientiousness (~ −1.74 σ) and slightly lower on Openness (a +0.45 score on the “Openness (low)” factor indicates somewhat reduced curiosity/novelty‑seeking), while Neuroticism is essentially average (~ 0 σ), implying typical emotional stability.
Overall: an outgoing, friendly communicator who fits Cluster 0 (Conscientiousness low) and would benefit most from clear structure, planning supports, and intentional variety to stretch openness.